# CP2 · Notebook 02 — Setup

Verificamos entorno, descargamos los 2 modelos desde HuggingFace y hacemos un test de inferencia para asegurarnos de que todo va.

**Tiempo**: ~2 min (primera vez, descarga ~50 MB de modelos). Siguientes ejecuciones: <30 s.

Al final debe imprimir `✅ Setup OK — listo para 03`.

## 1. Versiones de librerías

In [ ]:
import sys, platform
print(f'Python: {sys.version.split()[0]}  ({platform.system()} {platform.machine()})')
assert sys.version_info >= (3, 10), 'Necesitas Python 3.10+. Revisa tu venv.'

In [ ]:
import numpy as np, torch, transformers, timm, cv2, matplotlib
print(f'numpy        {np.__version__}')
print(f'torch        {torch.__version__}   (cuda: {torch.cuda.is_available()})')
print(f'transformers {transformers.__version__}')
print(f'timm         {timm.__version__}')
print(f'opencv       {cv2.__version__}')
print(f'matplotlib   {matplotlib.__version__}')

> Si `cuda: True` se usará GPU automáticamente — los tiempos del syllabus son **CPU**, así que tus números serán mejores. **Sin problema**, pero declara en el reporte que usaste GPU.

## 2. Imágenes — dataset combinado

Usamos:
- **CP1 lanes-subset** (14 imágenes Udacity CarND) — ya descargadas si hiciste CP1.
- **CP2 depth-extras** (5–7 imágenes de "depth challenge") — descargadas por `scripts/download_assets.py`.

In [ ]:
from pathlib import Path

CP1_DATA = Path('../../CP1-carriles/datasets/lanes-subset')
CP2_EXTRAS = Path('../datasets/cp2-depth-extras')

cp1_imgs = sorted(
    list(CP1_DATA.glob('**/*.png')) + list(CP1_DATA.glob('**/*.jpg'))
) if CP1_DATA.exists() else []
cp2_imgs = sorted(
    list(CP2_EXTRAS.glob('*.jpg')) + list(CP2_EXTRAS.glob('*.png'))
) if CP2_EXTRAS.exists() else []

print(f'CP1 lanes-subset: {len(cp1_imgs)} imágenes  ({CP1_DATA})')
print(f'CP2 depth-extras: {len(cp2_imgs)} imágenes  ({CP2_EXTRAS})')

missing = []
if len(cp1_imgs) < 14:
    missing.append(f'CP1 incompleto ({len(cp1_imgs)}/14). Ejecuta: cd ../../CP1-carriles && python scripts/download_assets.py --dataset-only')
if len(cp2_imgs) < 5:
    missing.append(f'CP2 extras incompleto ({len(cp2_imgs)}/5+). Ejecuta: python ../scripts/download_assets.py')

if missing:
    print('\n❌ Falta:')
    for m in missing: print(f'  - {m}')
else:
    print('\n✅ Imágenes OK')

## 3. Cargar MiDaS small

**Modelo**: `Intel/dpt-swinv2-tiny-256` — variante "small" de MiDaS empaquetada en `transformers` por Intel. ~22 MB.

In [ ]:
import time
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MIDAS_ID = 'Intel/dpt-swinv2-tiny-256'
print(f'Descargando/cargando {MIDAS_ID}...')
t0 = time.time()
midas_processor = AutoImageProcessor.from_pretrained(MIDAS_ID)
midas_model = AutoModelForDepthEstimation.from_pretrained(MIDAS_ID).eval()
print(f'  cargado en {time.time()-t0:.1f}s')
print(f'  params: {sum(p.numel() for p in midas_model.parameters())/1e6:.1f} M')

## 4. Cargar Depth Anything v2 small

**Modelo**: `depth-anything/Depth-Anything-V2-Small-hf` — variante small de DA v2 en formato HF. ~24 MB.

In [ ]:
DA_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
print(f'Descargando/cargando {DA_ID}...')
t0 = time.time()
da_processor = AutoImageProcessor.from_pretrained(DA_ID)
da_model = AutoModelForDepthEstimation.from_pretrained(DA_ID).eval()
print(f'  cargado en {time.time()-t0:.1f}s')
print(f'  params: {sum(p.numel() for p in da_model.parameters())/1e6:.1f} M')

## 5. Test rápido — 1 imagen sobre ambos modelos

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

test_img_path = cp1_imgs[0] if cp1_imgs else None
assert test_img_path, 'No hay imágenes test. Revisa la celda 2.'
img = Image.open(test_img_path).convert('RGB')
print(f'Test image: {test_img_path.name}  size={img.size}')

def infer(processor, model, image_pil):
    t0 = time.perf_counter()
    inputs = processor(images=image_pil, return_tensors='pt')
    with torch.no_grad():
        out = model(**inputs).predicted_depth
    ms = (time.perf_counter() - t0) * 1000
    return out[0].numpy(), ms

midas_depth, midas_ms = infer(midas_processor, midas_model, img)
da_depth,    da_ms    = infer(da_processor,    da_model,    img)

print(f'\nMiDaS:            output shape={midas_depth.shape}  range=[{midas_depth.min():.1f}, {midas_depth.max():.1f}]  t={midas_ms:.0f} ms')
print(f'Depth Anything:   output shape={da_depth.shape}     range=[{da_depth.min():.1f}, {da_depth.max():.1f}]  t={da_ms:.0f} ms')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(img); axes[0].set_title(f'Original · {test_img_path.name}'); axes[0].axis('off')
axes[1].imshow(midas_depth, cmap='plasma'); axes[1].set_title(f'MiDaS small ({midas_ms:.0f} ms)'); axes[1].axis('off')
axes[2].imshow(da_depth, cmap='plasma');    axes[2].set_title(f'Depth Anything v2 ({da_ms:.0f} ms)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

**Importante sobre los valores del depth map**:

- Ambos modelos devuelven **inverse depth** (relativo): valores **altos = más cerca**, valores bajos = más lejos.
- Los rangos son arbitrarios — no son metros. Pueden variar entre imágenes.
- Esto se trabaja en `04_relativa_vs_metrica.ipynb`.

Si el plot anterior muestra **azul/oscuro = lejos** y **amarillo/claro = cerca** con sentido geométrico, todo OK.

## 6. Cierre

In [ ]:
if not missing:
    print('✅ Setup OK — listo para 03_midas_basico.ipynb')
else:
    print('❌ Setup incompleto — revisa los warnings arriba')